In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-sonnet-4-5-20250929"

### Helper functions

In [3]:

from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
):
    params = {
        "model": model,
        "max_tokens": 10000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        file_metadata = get_metadata(id)
        file_content.write_to_file(file_metadata.filename)
    else:
        file_content.write_to_file(filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [4]:
import json

def show_response(response):
    """Messageオブジェクトをテキストで見やすく表示"""

    # メタ情報
    print(f"{'─'*60}")
    print(f"  Model      : {response.model}")
    print(f"  Stop reason: {response.stop_reason}")
    print(f"  Tokens     : input={response.usage.input_tokens}, output={response.usage.output_tokens}")
    print(f"{'─'*60}")

    # コンテンツブロック
    for i, block in enumerate(response.content):
        if block.type == "text":
            print(block.text)
        elif block.type == "tool_use":
            print(f"\n[Id    : {block.id}]")
            print(f"[Tool  : {block.name}]")
            print(json.dumps(block.input, indent=2, ensure_ascii=False))

    print(f"{'─'*60}")

### Run

In [7]:
file_metadata = upload("./documents/streaming.csv")
file_metadata

FileMetadata(id='file_011CYnY3fjpNPvzxuSYYAJbW', created_at=datetime.datetime(2026, 3, 6, 21, 28, 3, 773000, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False)

In [9]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

response = chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

In [10]:
response

Message(id='msg_01NB6VXv4LSjfTfP65dZYkbf', content=[TextBlock(citations=None, text="I'll help you perform a detailed churn analysis on the streaming.csv dataset. Let me start by exploring the data to understand its structure and then conduct a comprehensive analysis.", type='text'), ServerToolUseBlock(id='srvtoolu_01UdqKqnGnhwYWdPGGDUcs5a', input={'command': 'cd $INPUT_DIR && head -20 streaming.csv && wc -l streaming.csv'}, name='bash_code_execution', type='server_tool_use', caller={'type': 'direct'}), TextBlock(citations=None, text=None, type='bash_code_execution_tool_result', tool_use_id='srvtoolu_01UdqKqnGnhwYWdPGGDUcs5a', content={'type': 'bash_code_execution_result', 'stdout': 'UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSER_00002,Premium,41.4,Drama,5,9,45.7,3,17.

file_011CYnYMdAnfTZGCp1q3s14E
file_011CYnYQnnpdmmyquHiGWyPN
file_011CYnYSGr4Bo528ZgEaf5pb

In [11]:
#download_file("file_011CPYZqxoMSsfbrSzFw8j9X")

download_file("file_011CYnYMdAnfTZGCp1q3s14E")
download_file("file_011CYnYQnnpdmmyquHiGWyPN")
download_file("file_011CYnYSGr4Bo528ZgEaf5pb")